# 04. Field-of-View Optimization Concepts

Differential photometry needs good comparison stars inside the same frame
as the target. When the target sits near the edge of a sparse field, you
can rotate the camera (position angle) and nudge the pointing to pull more
usable comparison stars into the field of view.

This notebook demonstrates the geometry with a synthetic star field and a
coarse grid search. The production system (`src/muscat_db/fov.py`, the
`/fov` web page) does the same optimization against real Gaia DR3 stars
and per-instrument footprints.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.transforms as mtransforms

rng = np.random.default_rng(3)

## 1. A star field and a camera footprint

The target is at the origin of a local tangent plane (offsets in arcmin).
Comparison stars are useful only within a magnitude window: bright enough
for good signal, faint enough not to saturate. The camera footprint is a
square that we can translate and rotate.

In [ ]:
FOV_ARCMIN = 9.0          # square camera field, side length
MAG_LO, MAG_HI = 11.5, 15.5   # usable comparison-star magnitude window
N_FIELD = 60

# Stars scattered around the target, in arcmin offsets on the sky.
star_x = rng.uniform(-8, 8, N_FIELD)
star_y = rng.uniform(-8, 8, N_FIELD)
star_mag = rng.uniform(10.0, 18.0, N_FIELD)
usable = (star_mag >= MAG_LO) & (star_mag <= MAG_HI)

print(f"{usable.sum()} of {N_FIELD} stars fall in the comparison window")

In [ ]:
def in_footprint(cx, cy, pa_deg, half, px, py):
    """Which points (px, py) fall inside a square footprint.

    The square has side 2*half, is centered at (cx, cy), and rotated by
    pa_deg. We rotate the points into the box frame and test against an
    axis-aligned square.
    """
    a = np.deg2rad(pa_deg)
    dx, dy = px - cx, py - cy
    xr = np.cos(a) * dx + np.sin(a) * dy
    yr = -np.sin(a) * dx + np.cos(a) * dy
    return (np.abs(xr) <= half) & (np.abs(yr) <= half)


def draw_footprint(ax, cx, cy, pa_deg, half, **kw):
    rect = Rectangle((cx - half, cy - half), 2 * half, 2 * half,
                     fill=False, **kw)
    tr = mtransforms.Affine2D().rotate_deg_around(cx, cy, pa_deg) + ax.transData
    rect.set_transform(tr)
    ax.add_patch(rect)

## 2. Score a pointing

The score is the number of usable comparison stars inside the footprint,
subject to keeping the target itself in view with a safety margin. A
pointing that loses the target scores zero, however many comparisons it
catches.

In [ ]:
HALF = FOV_ARCMIN / 2
TARGET_MARGIN = 0.5   # arcmin the target must stay inside the edge

def score(cx, cy, pa_deg):
    if not in_footprint(cx, cy, pa_deg, HALF - TARGET_MARGIN,
                        np.array([0.0]), np.array([0.0]))[0]:
        return 0
    hit = in_footprint(cx, cy, pa_deg, HALF, star_x, star_y)
    return int(np.sum(hit & usable))

# Baseline: target centered, no rotation.
print(f"comparisons with default centered pointing: {score(0, 0, 0)}")

## 3. Grid search over offset and position angle

A coarse search over pointing offset (dx, dy) and position angle finds a
configuration that captures more comparison stars while keeping the target
framed. The real optimizer searches the same space with finer control and
instrument-specific constraints.

In [ ]:
offsets = np.linspace(-HALF + TARGET_MARGIN, HALF - TARGET_MARGIN, 9)
angles = np.arange(0, 90, 15)   # square has 90-deg symmetry

best = (score(0, 0, 0), 0.0, 0.0, 0.0)
for cx in offsets:
    for cy in offsets:
        for pa in angles:
            s = score(cx, cy, pa)
            if s > best[0]:
                best = (s, cx, cy, pa)

best_score, bx, by, bpa = best
print(f"best score : {best_score} comparison stars")
print(f"offset     : dx={bx:+.2f}, dy={by:+.2f} arcmin")
print(f"position PA: {bpa:.0f} deg")

## 4. Visualize the default and optimized pointings

Left: target centered, no rotation. Right: the grid-search result. Stars
in the comparison window are highlighted; the square is the camera
footprint.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
configs = [(0.0, 0.0, 0.0, "Default (centered)"),
           (bx, by, bpa, f"Optimized ({best_score} comps)")]

for ax, (cx, cy, pa, title) in zip(axes, configs):
    ax.scatter(star_x[~usable], star_y[~usable], s=20, color="0.7", label="other")
    ax.scatter(star_x[usable], star_y[usable], s=40, color="C0", label="usable comp")
    ax.scatter([0], [0], marker="*", s=260, color="C3", label="target", zorder=5)
    draw_footprint(ax, cx, cy, pa, HALF, color="C2", lw=2)
    ax.set_title(title)
    ax.set_xlabel("arcmin")
    ax.set_aspect("equal")
    ax.set_xlim(-9, 9)
    ax.set_ylim(-9, 9)
axes[0].set_ylabel("arcmin")
axes[0].legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

## Summary

- Comparison stars must share the frame with the target; a sparse field
  can be improved by rotating and offsetting the camera.
- The score trades comparison-star count against keeping the target
  safely framed.
- A small grid search over (offset, PA) already finds a better pointing.
- The production tool (`fov.py`, `/fov`) runs this against real Gaia DR3
  stars and true instrument footprints for observation planning.